In [1]:
import os, numpy as np, pandas as pd, librosa
from tqdm import tqdm

BASE = r"C:\Users\nabal\Documents\FYP"
SPLIT_DIR = os.path.join(BASE, "splits")
OUT_DIR   = os.path.join(BASE, "features_engineered_v1")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- audio/feature params ----------
SR = 22050
N_FFT = 2048
HOP = 512
FMIN_CQT = librosa.note_to_hz("C2")
BINS_PER_OCT = 12
N_BINS = 84  # 7 octaves
F0_MIN = librosa.note_to_hz("C2")
F0_MAX = librosa.note_to_hz("C7")

PITCHES = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]

def safe_stat(x):
    x = np.asarray(x)
    return dict(mean=float(np.nanmean(x)),
                std=float(np.nanstd(x)),
                min=float(np.nanmin(x)),
                max=float(np.nanmax(x)))

def extract_one(path):
    y, sr = librosa.load(path, sr=SR, mono=True)
    dur = len(y)/sr

    # ---- RMS & ZCR
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP).squeeze()
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP).squeeze()

    # ---- F0 (pyin) in Hz
    f0, vflag, _ = librosa.pyin(y, fmin=F0_MIN, fmax=F0_MAX, frame_length=N_FFT, hop_length=HOP)
    f0 = f0[vflag.astype(bool)]
    if f0.size == 0:
        # if pyin fails (too quiet), fill with NaNs so stats still compute
        f0 = np.array([np.nan])

    # ---- Chroma (CQT, 12 bins)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=HOP,
                                        fmin=FMIN_CQT, n_chroma=12,
                                        bins_per_octave=BINS_PER_OCT)
    # package features
    feat = {}
    # duration
    feat["duration_s"] = round(dur, 3)
    # rms & zcr
    for k,stats in [("rms", safe_stat(rms)), ("zcr", safe_stat(zcr))]:
        for s_name, val in stats.items():
            feat[f"{k}_{s_name}"] = val
    # f0 stats
    f0stats = safe_stat(f0)
    for s_name, val in f0stats.items():
        feat[f"f0_{s_name}"] = val
    # chroma stats per pitch
    cmean = chroma.mean(axis=1)
    cstd  = chroma.std(axis=1)
    for i,p in enumerate(PITCHES):
        feat[f"{p}_mean"] = float(cmean[i])
        feat[f"{p}_std"]  = float(cstd[i])
    return feat

def run_split(name):
    df = pd.read_csv(os.path.join(SPLIT_DIR, f"{name}.csv"))
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"Extract {name}"):
        feats = extract_one(r["file_path"])
        feats["file_path"] = r["file_path"]
        feats["maqam"] = r["maqam"]
        feats["reciter"] = r["reciter"]
        rows.append(feats)
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=False)
    print(f"✅ Saved {name} features → {os.path.join(OUT_DIR, f'{name}.csv')}  | shape {out.shape}")
    return out

train_f = run_split("train")
val_f   = run_split("val")
test_f  = run_split("test")

# quick sanity
print("\nMaqam counts (train):")
print(train_f["maqam"].value_counts())
print("\nFeature columns:", len(train_f.columns))


Extract train: 100%|██████████| 120/120 [40:13<00:00, 20.11s/it]


✅ Saved train features → C:\Users\nabal\Documents\FYP\features_engineered_v1\train.csv  | shape (120, 40)


Extract val: 100%|██████████| 25/25 [09:04<00:00, 21.79s/it]


✅ Saved val features → C:\Users\nabal\Documents\FYP\features_engineered_v1\val.csv  | shape (25, 40)


Extract test: 100%|██████████| 26/26 [07:58<00:00, 18.39s/it]

✅ Saved test features → C:\Users\nabal\Documents\FYP\features_engineered_v1\test.csv  | shape (26, 40)

Maqam counts (train):
maqam
Nahawand    40
Hijaz       40
Saba        40
Name: count, dtype: int64

Feature columns: 40
